# NMTK Quickstart

This notebook shows how to interact with the NMTK backend services from inside Jupyter.

Two helpers are pre-installed in this kernel:
- **`nmtk_client.lava`** — compile and run Lava/Loihi 2 spiking networks
- **`nmtk_client.suite`** — call any route on the suite_api gateway

Both read their service URLs from environment variables that are already set in the container:
- `LAVA_BACKEND_URL` → `http://lava-backend:8012`
- `SUITE_API_URL` → `http://suite_api:9000`

## 1 — Check services are reachable

In [ ]:
from nmtk_client import lava, suite

print("suite_api:", suite.health())
print("lava-backend:", lava.health())

## 2 — Run a Lava simulation

The workflow is always: **compile → run → stop**.

`run_config="sim"` uses software emulation (no Loihi chip needed).
Use `run_config="hw"` if a real Loihi 2 board is connected.

In [ ]:
# Define a small two-population LIF network
populations = [
    {"name": "input",  "size": 4, "threshold": 1.0},
    {"name": "output", "size": 4, "threshold": 1.0},
]
connections = [
    {"src": "input", "dst": "output", "weight": 0.6},
]

# Compile (may take 30-90 s on first call after container start)
session = lava.compile(populations, connections)
print("Session ID:", session)

In [ ]:
results = lava.run(session, steps=100)
print("Status:", results["status"])
print("Execution time (ms):", results.get("execution_time_ms"))
print("Spikes:", results["spikes"])

In [ ]:
# Plot spike raster
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(results["spikes"]), 1, figsize=(10, 3 * len(results["spikes"])))
if len(results["spikes"]) == 1:
    axes = [axes]
for ax, (pop_name, spike_train) in zip(axes, results["spikes"].items()):
    ax.plot(spike_train, drawstyle="steps-post")
    ax.set_title(f"Population: {pop_name}")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Spikes")
plt.tight_layout()
plt.show()

In [ ]:
lava.stop(session)
print("Session stopped.")

## 3 — Call suite_api

`suite.get()` and `suite.post()` let you reach any module route through the gateway.

In [ ]:
import json

health = suite.health()
print(json.dumps(health, indent=2))

## Next steps

- `lava.compile()` accepts any populations/connections your network needs.
- `suite.post("/api/neurocnl/...", {...})` reaches the NeuroStudio compiler routes.
- `suite.get("/api/neurobench/...")` reaches the NeuroBench benchmark routes.
- Clone this kernel with the **Environment Manager** (sidebar) to install extra packages without touching the base image.